# Pulizia del dataset (BRFSS 2015)

Il dataset è il questionario sanitario americano **BRFSS 2015**: 328.135 persone × 321 feature, target `_MICHD` (infarto / malattia coronarica, ~8.8% positivi).
Il 45% delle celle è NaN, ma i NaN **non sono casuali**: dipendono dalla struttura del questionario. Abbiamo individuato 4 cause:

| causa | esempio | colonne | trattamento |
|---|---|---|---|
| **1. Fisso vs cellulare**: due versioni del questionario | `CTELENUM` vs `CTELNUM1` | ~18 | eliminate (info sulla chiamata, non sulla salute) |
| **2. Intervista interrotta** (`DISPCODE`=1200, 15% delle righe) | `INCOME2`, `EXERANY2`, `_BMI5` | ~40 | NaN veri → imputazione (passo successivo) |
| **3. Domande condizionali**: chieste solo se la risposta precedente è "sì" | `AVEDRNK2` solo a chi beve | ~45 | il NaN significa "non applicabile" → 0 dove è una quantità |
| **4. Moduli opzionali** scelti dai singoli stati | modulo asma, vista, diabete | ~100 | quasi tutte eliminate o soglia sui NaN (passo successivo) |

Inoltre il BRFSS codifica le non-risposte con **codici numerici** (7/77/777 = "non so", 9/99/999 = "rifiuta") e lo zero con codici come 88 = "nessun giorno": vanno convertiti, altrimenti un modello lineare li legge come valori enormi.

In questo notebook:
1. eliminiamo le colonne inutili (identificativi, gestione della chiamata, pesi statistici, versione fisso/cellulare);
2. mettiamo **0 dove la domanda non è stata posta perché la risposta precedente era "no"** (zeri strutturali);
3. convertiamo i codici "nessuno" (88, 888, 555) in 0;
4. convertiamo i codici "non so / rifiuta" in NaN.

Le funzioni sono in `cleaning.py` (solo numpy).

In [ ]:
import numpy as np
from cleaning import (read_header, fill_structural_zeros, recode_none_to_zero,
                      find_special_codes, special_codes_to_nan)

DATA_PATH = "data/"   # cartella con x_train.csv, y_train.csv, x_test.csv

try:
    from helpers import load_csv_data
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
except ImportError:  # fallback senza helpers.py
    x_train = np.genfromtxt(DATA_PATH + "x_train.csv", delimiter=",", skip_header=1)
    x_test = np.genfromtxt(DATA_PATH + "x_test.csv", delimiter=",", skip_header=1)
    y_train = np.genfromtxt(DATA_PATH + "y_train.csv", delimiter=",", skip_header=1, usecols=1)
    train_ids, test_ids = x_train[:, 0].astype(int), x_test[:, 0].astype(int)
    x_train, x_test = x_train[:, 1:], x_test[:, 1:]

# nomi delle colonne: load_csv_data toglie già la colonna 'Id'
cols = read_header(DATA_PATH + "x_train.csv")
if len(cols) == x_train.shape[1] + 1:
    cols = cols[1:]
assert len(cols) == x_train.shape[1]

print("train:", x_train.shape, " test:", x_test.shape)
print(f"NaN iniziali: {np.isnan(x_train).mean():.1%}")

## 1. Colonne da eliminare a priori

- **Identificativi e date**: non contengono informazione sulla persona.
- **Gestione della chiamata** e **pesi statistici del sondaggio**: descrivono il campionamento, non la salute.
- **Gruppo 1, versione cellulare del questionario**: le colonne che esistono solo per chi è stato chiamato al cellulare (le corrispondenti del fisso sono già nella prima lista). Il loro NaN indica solo il tipo di telefono.
- **`HAREHAB1`** (riabilitazione dopo un infarto): è compilata solo per chi ha già avuto un infarto (target = 1 nel 100% dei 773 casi) → **leakage**.
- **Colonne grezze ridondanti**: ne esiste una versione già pulita, calcolata dal CDC:
  - `WEIGHT2`, `HEIGHT3` mescolano libbre/pollici e kg/cm → si usano `WTKG3`, `HTM4`, `_BMI5`;
  - `EXEROFT1/2`, `EXERHMM1/2` mescolano "volte a settimana" (1xx) e "volte al mese" (2xx) → si usano `PAFREQ*_`, `PADUR*_`, `PAMIN*_`;
  - `FLSHTMY2`, `HIVTSTD3` sono date (mese/anno).

In [ ]:
da_togliere = [
    # Identificativi e date
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    # Amministrative (gestione della chiamata)
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3',
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    # Pesi statistici del sondaggio
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT',
]

gruppo1_telefono = [
    # versione fisso: domande sulla famiglia / casa (solo chi è chiamato al fisso)
    'COLGHOUS', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'NUMPHON2',
    # versione cellulare
    'CTELNUM1', 'CELLFON2', 'CADULT', 'PVTRESD2', 'CCLGHOUS', 'CSTATE',
    'LANDLINE', 'HHADULT',
]

leakage = ['HAREHAB1']

ridondanti = ['WEIGHT2', 'HEIGHT3',                          # -> WTKG3, HTM4, _BMI5
              'EXEROFT1', 'EXERHMM1', 'EXEROFT2', 'EXERHMM2',  # -> PAFREQ*_, PADUR*_
              'FLSHTMY2', 'HIVTSTD3']                          # date

def drop_columns(X, cols, names):
    keep = [j for j, c in enumerate(cols) if c not in set(names)]
    return X[:, keep], [cols[j] for j in keep]

tutte = da_togliere + gruppo1_telefono + leakage + ridondanti
x_train, cols_tr = drop_columns(x_train, cols, tutte)
x_test,  cols_te = drop_columns(x_test,  cols, tutte)
assert cols_tr == cols_te
cols = cols_tr
print("colonne rimaste:", len(cols))

## 2. Zeri strutturali (domande condizionali)

Alcune domande sono poste solo a chi ha risposto "sì" a una domanda precedente (*madre*). Chi ha risposto "no" ha NaN nella domanda *figlia*, ma la risposta vera la conosciamo: **0**.
Esempio: `AVEDRNK2` ("quanti drink bevi in media?") è chiesta solo a chi ha bevuto nell'ultimo mese (`ALCDAY5` ≠ 888).

| persona | `ALCDAY5` (madre) | `AVEDRNK2` prima | dopo |
|---|---|---|---|
| A | 205 = ha bevuto 5 giorni | 2 | 2 |
| B | 888 = nessun giorno | NaN | **0** |
| C | NaN = intervista interrotta | NaN | NaN (resta un NaN vero) |

Mettiamo 0 **solo dove la madre dice esplicitamente "no"**:

| colonne figlie | madre | 0 se la madre vale |
|---|---|---|
| `AVEDRNK2`, `DRNK3GE5`, `MAXDRNKS` (alcol) | `ALCDAY5` | 888 = nessun giorno |
| `POORHLTH` (giorni limitati dalla salute) | `PHYSHLTH`, `MENTHLTH` | entrambe 88 = nessun giorno |
| attività fisica principale (`PAFREQ1_`, `PADUR1_`, `PAMIN11_`, `PA1MIN_`, `PAVIG11_`, `PA1VIGM_`, `_MINAC11`, `METVL11_`, `ACTIN11_`) | `EXERANY2` | 2 = non fa attività |
| seconda attività (`PAFREQ2_`, `PADUR2_`, `PAMIN21_`, `PAVIG21_`, `_MINAC21`, `METVL21_`, `ACTIN21_`) | `EXERANY2` / `EXRACT21` | 2 / 88 = nessuna seconda attività |

Caso speciale: `DIABAGE2` (età alla diagnosi del diabete) viene trasformata in **anni con il diabete** = `_AGE80` − `DIABAGE2`, con 0 per i non diabetici. Mettere 0 direttamente nell'età alla diagnosi non avrebbe senso. Tra i diabetici il tasso di infarto sale dal 17% (≤ 4 anni di diabete) al 29% (> 18 anni).

Le colonne madri restano nel dataset: così il modello distingue lo "0 vero" dallo "0 perché la domanda non è stata posta".

In [ ]:
def nan_frac(X, cols, names):
    ix = {c: i for i, c in enumerate(cols)}
    return {n: round(float(np.isnan(X[:, ix[n]]).mean()), 3) for n in names if n in ix}

check = ['AVEDRNK2', 'MAXDRNKS', 'POORHLTH', 'PA1MIN_', 'PAMIN21_', 'DIABAGE2']
prima = nan_frac(x_train, cols, check)

x_train = fill_structural_zeros(x_train, cols)
x_test  = fill_structural_zeros(x_test,  cols)

dopo = nan_frac(x_train, cols, check)
for c in check:
    print(f"{c:9s} NaN {prima[c]:.1%} -> {dopo[c]:.1%}")

## 3. Codici che significano "nessuno" → 0

Alcune colonne non usano lo 0 ma un codice: `88` = "nessun giorno / nessuno", `888` = "mai", `555` = "mai" (frutta e verdura).
Lasciati così, il modello leggerebbe per esempio "88 giorni di malessere" in un mese che ne ha 30.

- `PHYSHLTH`, `MENTHLTH`, `POORHLTH`, `CHILDREN`, `DRNK3GE5`: 88 → 0
- `ALCDAY5`, `STRENGTH`: 888 → 0
- `FRUITJU1`, `FRUIT1`, `FVBEANS`, `FVGREEN`, `FVORANG`, `VEGETAB1`: 555 → 0

Questo passo va fatto **dopo** il passo 2 (che legge i codici 88/888 delle madri) e **prima** del passo 4.

In [ ]:
x_train = recode_none_to_zero(x_train, cols)
x_test  = recode_none_to_zero(x_test,  cols)

## 4. Codici "non so / rifiuta" → NaN

Il BRFSS codifica le non-risposte con 7/77/777/7777 ("non so") e 9/99/999/9999 ("rifiuta"). La lunghezza del codice dipende da quante cifre ha la risposta.
Regola applicata a ogni colonna non continua, guardando il suo valore massimo:
- se il massimo è 9, 99, 999, … → i valori 9…9 e 7…7 diventano NaN;
- con una sola cifra il 7 è un codice solo se il 6 non compare: in `EMPLOY1` il 7 è "pensionato", una risposta valida;
- le colonne continue (BMI, minuti di attività, anni con diabete, …) vengono saltate;
- alcuni codici fuori schema sono indicati a mano (es. `_AGEG5YR` = 14, `_DRNKWEK` = 99900).

I codici vengono individuati **sul train** e applicati identici al test, così le due matrici sono trasformate nello stesso modo.\n`verbose=True` stampa, per ogni colonna, quali codici sono stati convertiti e su quante righe.

In [ ]:
# i codici si individuano sul TRAIN e si applicano identici al test
codes = find_special_codes(x_train, cols)
x_train = special_codes_to_nan(x_train, cols, codes, verbose=True)
x_test  = special_codes_to_nan(x_test,  cols, codes)

## Risultato

Dopo la pulizia i NaN rimasti sono solo **NaN veri**: interviste interrotte, "non so / rifiuta" e moduli opzionali non somministrati.
Il passo successivo è decidere, per questi:
- **colonne dei moduli opzionali** (> 80–90% NaN): eliminarle, tenendo eventualmente quelle utili come `INSULIN` (modulo diabete) con le categorie "non diabetico / sconosciuto / sì / no";
- **colonne rimaste**: imputare con la mediana (continue) o la moda (categoriche), calcolate solo sul train, più un indicatore di missing per le colonne in cui il NaN è frequente (es. `INCOME2`, 17%).

In [ ]:
nan_col = np.isnan(x_train).mean(axis=0)
print(f"colonne: {len(cols)}   NaN totali: {np.isnan(x_train).mean():.1%}")
for t in [0.0, 0.1, 0.5, 0.8, 0.9]:
    print(f"colonne con NaN > {t:.0%}: {(nan_col > t).sum()}")